RETRIEVER

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,get_response_synthesizer
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Settings

Settings.chunk_size = 128
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(verbose=True, similarity_top_k=2)
response_synthesizer = get_response_synthesizer(
    response_mode="compact",
)
query_engine = index.as_query_engine()

In [2]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()

In [3]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [4]:
retriever_result = {}
search_results = []

for count, question in enumerate(questions, start=1):
    search_this = f"{question}"
    search_result = retriever.retrieve(search_this)
    search_results = []
    
    for i in range(retriever.similarity_top_k):
        search_results.append(search_result[i].get_text())

    retriever_result[count] = search_results

print(retriever_result)

{1: ['In practical terms, fourth-dimensional manipulation might allow for the creation of superfluids with no friction, enhancing the efficiency of oxygen delivery within a liquid atmosphere. Additionally, the discovery of exotic matter with negative mass could enable novel technologies, such as advanced levitation and anti-gravity systems, further transforming human capabilities and infrastructure.\r\n\r\nThe journey from theoretical physics to practical application involves overcoming significant technological challenges. To access and manipulate the fourth dimension, we would need advanced technologies, including next-generation particle colliders and dimensional gateways.', 'The journey from theoretical physics to practical application involves overcoming significant technological challenges. To access and manipulate the fourth dimension, we would need advanced technologies, including next-generation particle colliders and dimensional gateways. These devices would require unprecede

GENERATOR

In [5]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import sys

torch.random.manual_seed(0)
model_id = "HuggingFaceH4/zephyr-7b-beta"
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype="auto", 
    trust_remote_code=True, max_length=500
)
assert torch.cuda.is_available(), "This model needs a GPU to run ..."
device = torch.cuda.current_device()
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Generator

In [7]:
responses=[]

In [8]:
for count, question in enumerate(questions, start=1):
    retriever_context_list = retriever_result.get(count, [])
    retriever_context = ' '.join(retriever_context_list)
    
    query = f"Consider the following context: '{retriever_context}'. Answer true or false question '{question}'. You have to answer using only True or False without any other explanation. Answer: "
    
    model_inputs = tokenizer(query, return_tensors="pt").to("cuda")
    generated_ids = model.generate(**model_inputs)
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
    aa = response.find("Answer:")
    if aa != -1:
        response = response[aa:]
    responses.append(response)
    print(count, response)

/home/jjh_test/anaconda3/envs/torch2/lib/python3.10/site-packages/transformers/generation/utils.py:1376: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


1 Answer:  False. The statement 'The discovery of a fourth spatial dimension was made through advancements in theoretical physics.' is false. The given context only discusses the potential implications and technological challenges of a hypothetical fourth spatial dimension, but does not claim that such a discovery has been made through advancements in theoretical physics.
2 Answer:  False. The new liquid atmosphere on Earth will not allow humans to breathe underwater without any devices. The text explains that terrestrial organisms would need to evolve mechanisms to extract oxygen from the liquid, potentially akin to gills in aquatic species. This suggests that humans would need to adapt and evolve new respiratory systems to breathe in the new liquid atmosphere, which is not currently possible. Therefore, the statement is false.
3 Answer:  True.
4 Answer:  False.
5 Answer:  False. The statement only mentions that advanced technologies, including particle colliders, would be needed to a

EVALUATOR

In [9]:
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [10]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [11]:
responses_str = []

In [12]:
for response in responses:
    response_str = str(response)
    if "True" in response_str:
        response_str = "True"
    elif "False" in response_str:
        response_str = "False"
    
    responses_str.append(response_str)

In [13]:
correct_count=0
number=0

for answer, response, response_str, question in zip(answers, responses, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 1 True or False: The discovery of a fourth spatial dimension was made through advancements in theoretical physics.
RESPONSE Answer:  False. The statement 'The discovery of a fourth spatial dimension was made through advancements in theoretical physics.' is false. The given context only discusses the potential implications and technological challenges of a hypothetical fourth spatial dimension, but does not claim that such a discovery has been made through advancements in theoretical physics.
CORRECT ANSWER True

<<wrong>>
 5 True or False: Scientists plan to use particle colliders to manipulate the fourth dimension.
RESPONSE Answer:  False. The statement only mentions that advanced technologies, including particle colliders, would be needed to access and manipulate the fourth dimension, not that scientists plan to use them for this purpose.
CORRECT ANSWER True

<<wrong>>
 6 True or False: The discovery of a fourth spatial dimension led to the concept of a liquid atmosphere r

In [14]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

Total Questions: 75
Correct Answers: 62
Accuracy: 82.67%
